# Fine-tune the oil palm crown detector on mature palms (Peru)

The released detector ([model card](https://huggingface.co/dexsholex/oil-palm-crown-detector)) learned from
young palms in drone orthomosaics of one plantation in West Africa. On close-up photos of mature palms with
overlapping crowns from Peru it finds at best about half of them. This notebook measures how much a few
hundred labelled Peru photos fix that.

It runs from top to bottom with no changes:

1. Downloads the released model (run 9) from Hugging Face and the public Peru dataset from Mendeley Data.
2. Splits the Peru photos into training, validation and test sets, keeping neighbouring photos (which can
   show the same palm) on the same side.
3. Scores the released model on the Peru photos (before).
4. Fine-tunes it with four settings (two input sizes, two learning rates) and keeps the one that scores best
   on the validation photos.
5. Scores the kept model once on the test photos (after) and saves everything in
   `peru_finetune_results.zip`.

**Settings.** Kaggle: in the right-hand panel under *Session options*, set *Accelerator* to *GPU T4 x2*
(one GPU is used) and turn *Internet* on. Google Colab: Runtime → Change runtime type → *T4 GPU*. A full run
takes well under an hour on a T4.

**Data.** *Oil Palm Tree Detection for Anomaly Identification*, Dominguez Meza & Rituay, Mendeley Data,
[doi:10.17632/nh7d23dgnw.1](https://doi.org/10.17632/nh7d23dgnw.1), CC BY 4.0. Its healthy and anomalous
palm classes are counted together as one class, `oil_palm`.

Code: [github.com/Dexsholex/oil-palm-crown-detection](https://github.com/Dexsholex/oil-palm-crown-detection)
(AGPL-3.0). Cite the model as [doi:10.5281/zenodo.23704654](https://doi.org/10.5281/zenodo.23704654).

In [ ]:
# The Ultralytics version the released model was checked with.
!pip install -q ultralytics==8.4.178

In [ ]:
import csv, hashlib, io, json, os, random, re, shutil, urllib.request, zipfile
from pathlib import Path

import numpy as np
import torch
import ultralytics
from PIL import Image, ImageDraw
from ultralytics import YOLO

SMOKE = os.environ.get("PALM_SMOKE") == "1"  # tiny CPU run used to test this notebook; leave unset

# Kaggle keeps /kaggle/working as the notebook's output; Colab works in /content.
WORK = next((Path(p) for p in ["/kaggle/working", "/content"] if Path(p).is_dir()), Path("peru_work"))
WORK = Path(os.environ.get("PALM_WORK", WORK))
RESULTS = WORK / "results"

WEIGHTS_URL = "https://huggingface.co/dexsholex/oil-palm-crown-detector/resolve/main/oil_palm_crown.pt"
WEIGHTS_MD5 = "b219fd1f5a44575061196bfe0f9635dd"
DATA_URL = ("https://data.mendeley.com/public-files/datasets/nh7d23dgnw/files/"
            "c4cba156-35e5-415a-a8bc-9054c5e41eda/file_downloaded")
DATA_MD5 = "54070f603055e8166295a4ffbba2c55a"

SEED = 0
BLOCK = 20                        # photos numbered in the same block of 20 go to the same split
TEST_SHARE, VAL_SHARE = 0.15, 0.15
NEAR_DUPLICATE = 0.5              # a validation or test photo this similar to a training photo moves to training
IMAGE_SIZES = [320, 640]          # 320 px shrinks Peru palms to about the training palm size; 640 keeps detail
LEARNING_RATES = [0.0002, 0.001]  # run 9's rate and a 5x higher one
BASELINE_SIZES = [256, 320, 480, 640]
EPOCHS, PATIENCE, BATCH, WORKERS, FRACTION = 100, 25, 16, 2, 1.0
CONF, NMS_IOU = 0.25, 0.45        # the production pipeline's settings
DEVICE = 0

if SMOKE:
    IMAGE_SIZES, LEARNING_RATES, BASELINE_SIZES = [320], [0.001], [320]
    EPOCHS, PATIENCE, BATCH, WORKERS, FRACTION, DEVICE = 1, 1, 4, 0, 0.3, "cpu"
elif not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Kaggle: Settings → Accelerator → GPU T4 x2. "
                       "Colab: Runtime → Change runtime type → T4 GPU.")

if RESULTS.exists():
    shutil.rmtree(RESULTS)  # start clean when the notebook is re-run
RESULTS.mkdir(parents=True)
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("Working in", WORK.resolve(), "| GPU:", GPU)

In [ ]:
def fetch(url, dest, md5):
    """Download once, then check the file is exactly the one this notebook was written for."""
    if not dest.exists():
        print("Downloading", dest.name, "...")
        part = dest.with_suffix(".part")
        request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request) as response, open(part, "wb") as out:
            shutil.copyfileobj(response, out)
        part.rename(dest)
    digest = hashlib.md5(dest.read_bytes()).hexdigest()
    if digest != md5:
        raise RuntimeError(f"{dest.name} has checksum {digest}, expected {md5}. Delete it and re-run.")
    return dest


weights_run9 = fetch(WEIGHTS_URL, Path(os.environ.get("PALM_WEIGHTS", WORK / "oil_palm_crown_run9.pt")), WEIGHTS_MD5)
peru_zip = fetch(DATA_URL, Path(os.environ.get("PALM_PERU_ZIP", WORK / "peru_oil_palm.zip")), DATA_MD5)

archive = zipfile.ZipFile(peru_zip)
labels, size = {}, {}  # photo path in the zip -> palm boxes (x_min, y_min, x_max, y_max) and (width, height)
for part in ["train", "valid", "test"]:
    with archive.open(f"{part}/_annotations.csv") as fh:
        for row in csv.DictReader(io.TextIOWrapper(fh, encoding="utf-8")):
            path = f"{part}/{row['filename']}"
            labels.setdefault(path, []).append(tuple(float(row[k]) for k in ("xmin", "ymin", "xmax", "ymax")))
            size[path] = (int(row["width"]), int(row["height"]))
photos = sorted(n for n in archive.namelist() if n.endswith(".jpg"))
for p in photos:
    if p not in size:  # a photo with no palms has no row in the label file
        size[p] = Image.open(archive.open(p)).size


def photo_number(path):
    return int(re.search(r"imgPalm(\d+)_", path).group(1))


print(f"{len(photos)} photos, {sum(len(labels.get(p, [])) for p in photos)} labelled palms")

## A split without leaks

The published split assigns photos to train, valid and test at random. The photos are numbered in the order
they were taken, and neighbouring numbers can show the same palm (photos 200 and 201, for example), so a
random split can put the same palm in both training and test, and the test score then looks better than it
is. Here whole blocks of 20 consecutive numbers go to one split. As a final check, any validation or test
photo that looks almost the same as a training photo is moved to training.

In [ ]:
blocks = sorted({photo_number(p) // BLOCK for p in photos})
random.Random(SEED).shuffle(blocks)
split_of_block, filled = {}, {"test": 0, "val": 0, "train": 0}
for b in blocks:
    target = ("test" if filled["test"] < TEST_SHARE * len(photos)
              else "val" if filled["val"] < VAL_SHARE * len(photos) else "train")
    split_of_block[b] = target
    filled[target] += sum(1 for p in photos if photo_number(p) // BLOCK == b)
split = {p: split_of_block[photo_number(p) // BLOCK] for p in photos}


def thumbnail(path):
    """Small grey thumbnail, scaled so that the dot product of two thumbnails is their correlation."""
    grey = np.asarray(Image.open(archive.open(path)).convert("L").resize((64, 48), Image.BILINEAR), np.float32)
    grey = (grey - grey.mean()) / (grey.std() + 1e-6)
    return grey.ravel() / np.sqrt(grey.size)


similarity = np.stack([thumbnail(p) for p in photos])
similarity = similarity @ similarity.T


def near_duplicates(assignment):
    """Validation/test photos whose closest training photo is at least NEAR_DUPLICATE similar."""
    in_train = np.array([assignment[p] == "train" for p in photos])
    return {p: float(similarity[i, in_train].max()) for i, p in enumerate(photos)
            if not in_train[i] and similarity[i, in_train].max() >= NEAR_DUPLICATE}


published = {p: p.split("/")[0] for p in photos}
published_leaks = near_duplicates(published)
moved = {}
while leaks := near_duplicates(split):  # repeat: a photo moved to training can make another one a near-duplicate
    for p, sim in leaks.items():
        moved[p] = (split[p], sim)
        split[p] = "train"

DATASET = WORK / "peru_yolo"
if DATASET.exists():
    shutil.rmtree(DATASET)
for p in photos:
    folder = DATASET / split[p]
    (folder / "images").mkdir(parents=True, exist_ok=True)
    (folder / "labels").mkdir(parents=True, exist_ok=True)
    stem, (w, h) = Path(p).stem, size[p]
    (folder / "images" / f"{stem}.jpg").write_bytes(archive.read(p))
    lines = [f"0 {(x0 + x1) / 2 / w:.6f} {(y0 + y1) / 2 / h:.6f} {(x1 - x0) / w:.6f} {(y1 - y0) / h:.6f}"
             for x0, y0, x1, y1 in labels.get(p, [])]
    (folder / "labels" / f"{stem}.txt").write_text("\n".join(lines))
DATA_YAML = DATASET / "data.yaml"
DATA_YAML.write_text(f"path: {DATASET.resolve().as_posix()}\ntrain: train/images\nval: val/images\n"
                     "test: test/images\nnames:\n  0: oil_palm\n")

with open(RESULTS / "peru_split.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["photo", "photo_number", "published_split", "split", "palms", "note"])
    for p in photos:
        note = (f"moved from {moved[p][0]}: similarity {moved[p][1]:.2f} to a training photo" if p in moved else "")
        writer.writerow([Path(p).name, photo_number(p), published[p], split[p], len(labels.get(p, [])), note])

summary = {"split": {s: {"photos": sum(split[p] == s for p in photos),
                         "palms": sum(len(labels.get(p, [])) for p in photos if split[p] == s)}
                     for s in ["train", "val", "test"]}}
print("New split:", summary["split"])
print(f"Published split: {len(published_leaks)} validation/test photo(s) nearly identical to a training photo")
print(f"New split: {len(moved)} photo(s) moved to training as near-duplicates")

In [ ]:
def score(weights, part, imgsz, name):
    """Ultralytics' standard detection scores on one split."""
    m = YOLO(str(weights)).val(data=str(DATA_YAML), split=part, imgsz=imgsz, batch=BATCH, device=DEVICE,
                               workers=WORKERS, plots=False, verbose=False,
                               project=str(RESULTS / "val"), name=name, exist_ok=True)
    return {"precision": m.box.mp, "recall": m.box.mr, "mAP50": m.box.map50, "mAP50_95": m.box.map}


def iou_matrix(a, b):
    x0, y0 = np.maximum(a[:, None, 0], b[None, :, 0]), np.maximum(a[:, None, 1], b[None, :, 1])
    x1, y1 = np.minimum(a[:, None, 2], b[None, :, 2]), np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x1 - x0, 0, None) * np.clip(y1 - y0, 0, None)
    area = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter)


def count_matches(pred, conf, gt, thresh=0.5):
    """Greedy one-to-one matching, highest-confidence detection first (as in scripts/evaluate_test_split.py)."""
    if len(pred) == 0 or len(gt) == 0:
        return 0
    used, matched = set(), 0
    for row in iou_matrix(pred[np.argsort(-conf)], gt):
        for j in np.argsort(-row):
            if row[j] < thresh:
                break
            if j not in used:
                used.add(j)
                matched += 1
                break
    return matched


def count_check(weights, imgsz, part="test"):
    """Counts at the pipeline's settings (confidence 0.25, NMS IoU 0.45), photo by photo."""
    model = YOLO(str(weights))
    t = dict(labelled=0, detected=0, matched=0, found=0, on_palm=0, abs_error=0, empty=0)
    images = sorted((DATASET / part / "images").glob("*.jpg"))
    boxes = {}
    for img in images:
        r = model.predict(str(img), imgsz=imgsz, conf=CONF, iou=NMS_IOU, device=DEVICE, verbose=False)[0]
        pred, conf = r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()
        h, w = r.orig_shape
        lab = np.loadtxt(DATASET / part / "labels" / f"{img.stem}.txt", ndmin=2)
        gt = (np.stack([(lab[:, 1] - lab[:, 3] / 2) * w, (lab[:, 2] - lab[:, 4] / 2) * h,
                        (lab[:, 1] + lab[:, 3] / 2) * w, (lab[:, 2] + lab[:, 4] / 2) * h], axis=1)
              if lab.size else np.zeros((0, 4)))
        cx, cy = (pred[:, 0] + pred[:, 2]) / 2, (pred[:, 1] + pred[:, 3]) / 2
        inside = ((cx[:, None] >= gt[None, :, 0]) & (cx[:, None] <= gt[None, :, 2]) &
                  (cy[:, None] >= gt[None, :, 1]) & (cy[:, None] <= gt[None, :, 3]))
        t["labelled"] += len(gt)
        t["detected"] += len(pred)
        t["matched"] += count_matches(pred, conf, gt)
        t["found"] += int(inside.any(axis=0).sum())   # labelled palms with a detection centred on them
        t["on_palm"] += int(inside.any(axis=1).sum())  # detections centred on a labelled palm
        t["abs_error"] += abs(len(pred) - len(gt))
        t["empty"] += int(len(pred) == 0)
        boxes[img.stem] = pred
    return {"photos": len(images), "labelled_palms": t["labelled"], "detections": t["detected"],
            "precision_iou50": t["matched"] / max(t["detected"], 1),
            "recall_iou50": t["matched"] / max(t["labelled"], 1),
            "palms_found": t["found"] / max(t["labelled"], 1),
            "detections_on_a_palm": t["on_palm"] / max(t["detected"], 1),
            "count_error_pct": 100 * (t["detected"] - t["labelled"]) / max(t["labelled"], 1),
            "mean_abs_error_per_photo": t["abs_error"] / max(len(images), 1),
            "photos_without_detection": t["empty"]}, boxes

## Before: the released model

The released model does best when palms look about as large as in its training tiles, so its input size is
chosen on the validation photos (a smaller input shrinks the palms).

In [ ]:
baseline_val = {s: score(weights_run9, "val", s, f"run9_val_{s}px") for s in BASELINE_SIZES}
for s, m in baseline_val.items():
    print(f"Released model at {s} px, validation: mAP50 {m['mAP50']:.3f}, recall {m['recall']:.3f}")
baseline_size = max(BASELINE_SIZES, key=lambda s: baseline_val[s]["mAP50"])
print("Best input size for the released model:", baseline_size)

## Fine-tuning

Starts from the released model's weights and uses the same recipe as its training (AdamW, rotations up to
180°, flips, mosaic). Four settings are tried and the one with the best validation mAP@0.5 is kept. The test
photos are not touched here.

In [ ]:
trials = []
for imgsz in IMAGE_SIZES:
    for lr in LEARNING_RATES:
        name = f"peru_{imgsz}px_lr{lr:g}"
        YOLO(str(weights_run9)).train(
            data=str(DATA_YAML), imgsz=imgsz, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, fraction=FRACTION,
            optimizer="AdamW", lr0=lr, lrf=0.01, momentum=0.937, weight_decay=0.0005, warmup_epochs=2.0,
            degrees=180, translate=0.1, scale=0.5, shear=0.0, perspective=0.0, flipud=0.5, fliplr=0.5,
            mosaic=1.0, mixup=0.0, close_mosaic=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
            cache="ram", seed=SEED, deterministic=True, device=DEVICE, workers=WORKERS,
            plots=not SMOKE, project=str(RESULTS / "train"), name=name, exist_ok=True)
        best = RESULTS / "train" / name / "weights" / "best.pt"
        trials.append({"name": name, "imgsz": imgsz, "lr0": lr, "weights": best,
                       "val": score(best, "val", imgsz, f"{name}_val")})
        print(f"{name}: validation mAP50 {trials[-1]['val']['mAP50']:.3f}")
chosen = max(trials, key=lambda t: t["val"]["mAP50"])
FINAL = RESULTS / "oil_palm_crown_peru.pt"
shutil.copy(chosen["weights"], FINAL)
print("Kept", chosen["name"])

## After: one look at the test photos

Both models are scored on the same held-out test photos: Ultralytics' standard scores, then counts at the
production pipeline's settings (confidence 0.25). "Palms found" counts a labelled palm as found when a
detection is centred on it, the same lenient measure as the earlier Peru evaluation in the repository.

In [ ]:
models = [(f"Released model (run 9), {baseline_size} px", weights_run9, baseline_size, "run9_test"),
          (f"Fine-tuned on Peru, {chosen['imgsz']} px, lr {chosen['lr0']:g}", FINAL, chosen["imgsz"], "peru_test")]
test_rows, test_boxes = [], {}
for label, weights, imgsz, tag in models:
    counts, test_boxes[label] = count_check(weights, imgsz)
    test_rows.append({"model": label, "imgsz": imgsz, **score(weights, "test", imgsz, tag), **counts})

print("| Model | Precision | Recall | mAP@0.5 | mAP@0.5:0.95 | Palms found | Count vs labels | Photos with no detection |")
print("|---|---|---|---|---|---|---|---|")
for r in test_rows:
    print(f"| {r['model']} | {r['precision']:.3f} | {r['recall']:.3f} | {r['mAP50']:.3f} | {r['mAP50_95']:.3f} | "
          f"{100 * r['palms_found']:.0f}% | {r['detections']} vs {r['labelled_palms']} ({r['count_error_pct']:+.1f}%) | "
          f"{r['photos_without_detection']} of {r['photos']} |")

# Side-by-side pictures for the test photos with the most palms: labels, released model, fine-tuned model.
by_stem = {Path(p).stem: p for p in photos}
examples = sorted((s for s in test_boxes[models[0][0]]), key=lambda s: (-len(labels.get(by_stem[s], [])), s))[:4]
(RESULTS / "examples").mkdir()
for stem in examples:
    panels = []
    for boxes, colour in [(np.array(labels.get(by_stem[stem], [])).reshape(-1, 4), "#00e676"),
                          (test_boxes[models[0][0]][stem], "#ffd600"), (test_boxes[models[1][0]][stem], "#ffd600")]:
        im = Image.open(archive.open(by_stem[stem])).convert("RGB")
        draw = ImageDraw.Draw(im)
        for x0, y0, x1, y1 in boxes:
            draw.rectangle([x0, y0, x1, y1], outline=colour, width=6)
        panels.append(im.resize((im.width // 2, im.height // 2)))
    sheet = Image.new("RGB", (sum(p.width for p in panels) + 20, panels[0].height), "white")
    for i, panel in enumerate(panels):
        sheet.paste(panel, (i * (panel.width + 10), 0))
    sheet.save(RESULTS / "examples" / f"{stem}.jpg", quality=85)

summary.update({
    "settings": {"seed": SEED, "block": BLOCK, "near_duplicate": NEAR_DUPLICATE, "image_sizes": IMAGE_SIZES,
                 "learning_rates": LEARNING_RATES, "epochs": EPOCHS, "patience": PATIENCE, "batch": BATCH,
                 "fraction": FRACTION, "conf": CONF, "nms_iou": NMS_IOU, "smoke_test": SMOKE},
    "published_split_near_duplicates": published_leaks,
    "moved_to_training": {p: {"from": s, "similarity": sim} for p, (s, sim) in moved.items()},
    "baseline_validation": baseline_val,
    "trials": [{k: v for k, v in t.items() if k != "weights"} for t in trials],
    "chosen": chosen["name"],
    "test": test_rows,
    "versions": {"ultralytics": ultralytics.__version__, "torch": torch.__version__, "gpu": GPU},
})
(RESULTS / "summary.json").write_text(json.dumps(summary, indent=2, default=float))

bundle = WORK / "peru_finetune_results.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as out:
    for f in sorted(RESULTS.rglob("*")):
        if f.is_file() and (f.suffix != ".pt" or f == FINAL):  # keep only the chosen model's weights
            out.write(f, f.relative_to(RESULTS))
print(f"Saved {bundle} ({bundle.stat().st_size / 1e6:.0f} MB)")
try:
    from google.colab import files  # on Colab, start the download straight away
    files.download(str(bundle))
except ImportError:
    pass

## Next

Download `peru_finetune_results.zip` (Kaggle: the *Output* panel on the right; Colab: it downloads by
itself). It holds the fine-tuned model `oil_palm_crown_peru.pt`, `summary.json` with every score above,
the split (`peru_split.csv`), training curves and example pictures.

One question remains that this notebook cannot answer: did learning Peru's palms make the model worse on
the West African plantation it was built for? That test needs the original labelled tiles, which are
private, so it runs on the author's computer with `scripts/evaluate_test_split.py`.

It did: the fine-tuned model's West African test mAP@0.5 fell from 0.844 to 0.004. `finetune_joint.ipynb`
trains on both sites together and avoids this; results are in the repository's `evaluation/` folder.